In [5]:
import os
import sys
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

from utils.summary import get_model_stats

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm  
from torchvision.models import resnet34

import warnings
warnings.filterwarnings('ignore')

In [11]:
import torch
import torch.nn as nn
from torchvision.models import resnet34

class UNet(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(UNet, self).__init__()
        # Load a pretrained ResNet encoder
        self.encoder = resnet34(pretrained=pretrained)
        self.base_layers = list(self.encoder.children())

        # Encoder layers
        self.enc1 = nn.Sequential(*self.base_layers[:3])  # Conv1 + BN + ReLU
        self.enc2 = nn.Sequential(*self.base_layers[3:5])  # MaxPool + Layer1
        self.enc3 = self.base_layers[5]  # Layer2
        self.enc4 = self.base_layers[6]  # Layer3
        self.enc5 = self.base_layers[7]  # Layer4

        # Decoder with extra upsampling to restore original size
        self.up4 = self._upsample_block(512, 256)
        self.up3 = self._upsample_block(256, 128)
        self.up2 = self._upsample_block(128, 64)
        self.up1 = self._upsample_block(64, 64)
        self.up_final = nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1)

        # Final output layer
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def _upsample_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        # Decoder with skip connections
        d4 = self.up4(e5) + e4
        d3 = self.up3(d4) + e3
        d2 = self.up2(d3) + e2
        d1 = self.up1(d2) + e1
        
        # Final upsampling to restore original size
        d_final = self.up_final(d1)

        # Final output
        out = self.final_conv(d_final)
        return out

# Example usage
model = UNet(num_classes=1, pretrained=True)  # Adjust num_classes for your task
input_tensor = torch.randn(1, 3, 256, 256)  # Example input
output = model(input_tensor)
print("Output shape:", output.shape)
get_model_stats(model, input_tensor.shape)

Output shape: torch.Size([1, 1, 256, 256])


Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::add encountered 4 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
encoder.avgpool, encoder.fc


{'flops': 6050349056, 'params': 24231849}

In [ ]:
from data.voc import get_voc_pipeline

train_loader, val_loader, test_loader = get_voc_pipeline(batch_size=4)
sample_x, sample_y = next(iter(train_loader))
print(sample_x.shape)
print(sample_y.shape)

In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyperparameters
batch_size = 128  # Adjust based on GPU memory
initial_lr = 0.01  # Starting learning rate
max_lr = 0.1  # Maximum learning rate after warmup
epochs = 50
warmup_epochs = 5  # Number of warmup epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data preparation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalization
])

train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model (using a ResNet50 pretrained model as an example)
from torchvision.models import resnet50
model = resnet50(pretrained=False, num_classes=10)  # Adjust number of classes
model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=0.9, weight_decay=5e-4)

# Learning rate scheduler with warmup
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # Linear warmup
    return 1.0

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# Training loop
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

# Evaluation loop
def evaluate(model, loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
    accuracy = correct / len(loader.dataset)
    return test_loss / len(loader), accuracy

# Main training script
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_accuracy = evaluate(model, test_loader, criterion)
    scheduler.step()  # Update learning rate
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Training complete.")
